# 02 · Calidad de datos y trazabilidad de la limpieza

**Objetivo.** Documentar qué se eliminó y por qué, cuánto dato está censurado
(bajo el límite de detección) y cómo cambia el resultado según el método de
imputación de la censura.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "reports" / "tables"
FIGURES = ROOT / "reports" / "figures"

## 1. Bitácora de limpieza (reglas R1-R10)

In [ ]:
log = pd.read_csv(TABLES / "log_limpieza.csv", encoding="utf-8-sig")
log

**Lectura.** La regla R5 elimina los *placeholders* (`valor = 0` sin unidad ni
signo), que no son mediciones sino celdas vacías codificadas. Al quitarlas
desaparecen además todos los duplicados conflictivos.

## 2. Censura por parámetro

In [ ]:
cens = pd.read_csv(TABLES / "censura_por_parametro.csv", encoding="utf-8-sig")
cens.sort_values("pct_censurado", ascending=False).head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
top = cens.sort_values("pct_censurado", ascending=False).head(20).iloc[::-1]
ax.barh(top["parametro_estandar"], top["pct_censurado"], color="#fdae61")
ax.set_xlabel("% de valores censurados (< límite de detección)")
ax.set_title("Parámetros con mayor censura")
plt.tight_layout()
plt.show()

**Lectura.** Los parámetros con censura > 80 % se tratan como binarios
(detectado / no detectado); el resto se imputa como LD/2, que es la práctica
habitual en química ambiental.

## 3. Sensibilidad al método de censura

In [ ]:
sens_cens = pd.read_csv(TABLES / "sensibilidad_censura.csv", encoding="utf-8-sig")
sens_cens

## 4. Sensibilidad al fondo geoquímico

In [ ]:
sens_fondo = pd.read_csv(TABLES / "sensibilidad_fondo.csv", encoding="utf-8-sig")
sens_fondo

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
escenarios = sens_fondo['escenario'].tolist()
ax.bar(escenarios, sens_fondo['pct_PLI_mayor_1'], color=['#2c7bb6', '#333333', '#fdae61', '#d7191c'],
       edgecolor='k', linewidth=0.4)
for i, (pct, med) in enumerate(zip(sens_fondo['pct_PLI_mayor_1'], sens_fondo['mediana_PLI'])):
    ax.text(i, pct + 1.5, f'{pct:.0f} %' + chr(10) + f'PLI med {med:.2f}', ha='center', fontsize=8)
ax.set_ylabel('% de muestras con PLI > 1')
ax.set_title('Sensibilidad del PLI al fondo geoquimico elegido')
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

sens_fondo

**Lectura.** El PLI es sensible al fondo elegido, por eso el escenario principal
usa el P25 local (fondo geogénico de la cuenca) y se reportan P10/P50 y la
corteza de Turekian & Wedepohl como análisis de sensibilidad.